# Agent Chat UI (OSS)

Agent Chat UI는 LangGraph로 구축한 에이전트를 웹 브라우저에서 대화형으로 테스트할 수 있는 오픈소스 채팅 인터페이스입니다. LangGraph 서버와 Agent Chat UI를 연결하면, 도구 호출 과정, 멀티턴 대화, 스트리밍 응답 등을 시각적으로 확인하며 에이전트를 개발할 수 있습니다.

이 튜토리얼에서는 Tavily 검색 도구를 사용하는 ReAct 에이전트를 구축하고, `langgraph dev` 명령으로 로컬 서버를 실행한 뒤, Agent Chat UI를 연결하여 에이전트와 대화하는 전체 과정을 다룹니다.

> 참고 문서: [Agent Chat UI GitHub](https://github.com/langchain-ai/agent-chat-ui)

![Agent Chat UI Architecture](assets/agent-chat-ui-architecture.png)

## 환경 설정

LangGraph 에이전트를 실행하기 위해 필요한 환경을 설정합니다. `.env` 파일에서 API 키를 로드하고, LangSmith를 통해 에이전트의 실행 과정을 추적할 수 있도록 설정합니다.

아래 코드는 환경 변수를 로드하고 LangSmith 프로젝트를 설정합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv(override=True)
logging.langsmith("LangGraph-V1-Tutorial")

## langgraph.json 설정 파일

`langgraph.json`은 LangGraph 서버의 설정 파일입니다. 이 파일은 서버가 어떤 그래프를 로드할지, 환경 변수 파일의 경로, Python 버전, 의존성 패키지 등을 정의합니다. `langgraph dev` 명령은 이 파일을 읽어 에이전트 서버를 구동합니다.

주요 설정 항목은 다음과 같습니다:

| 항목 | 설명 |
|:---|:---|
| `graphs` | 그래프 이름과 파일 경로 매핑 (예: `"agent": "./agent.py:graph"`) |
| `env` | 환경 변수 파일 경로 (`.env` 파일 위치) |
| `python_version` | 사용할 Python 버전 |
| `dependencies` | 의존성 패키지 경로 (상위 디렉토리의 `pyproject.toml` 포함) |

아래 코드는 현재 디렉토리의 `langgraph.json` 파일 내용을 확인합니다.

In [ ]:
import json

# langgraph.json 파일 내용 확인
with open("langgraph.json", "r") as f:
    config = json.load(f)

print(json.dumps(config, indent=2, ensure_ascii=False))

## agent.py 에이전트 정의

`agent.py`는 LangGraph 서버에서 실행할 에이전트를 정의하는 파일입니다. 이 파일에서는 `create_agent` 함수를 사용하여 Tavily 검색 도구가 연동된 ReAct 에이전트를 생성합니다. `MemorySaver` 체크포인터를 통해 대화 히스토리를 유지하므로 멀티턴 대화가 가능합니다.

`langgraph.json`에서 `"agent": "./agent.py:graph"` 로 지정했으므로, `agent.py` 모듈의 `graph` 변수가 서버에 로드됩니다.

아래 코드는 `agent.py` 파일의 소스 코드를 확인합니다.

In [ ]:
# agent.py 소스 코드 확인
with open("agent.py", "r") as f:
    print(f.read())

## 에이전트 로컬 테스트

LangGraph 서버에 배포하기 전에, 노트북 환경에서 에이전트가 정상적으로 동작하는지 테스트합니다. `agent.py`에서 정의한 그래프를 직접 import하여 시각화하고 실행해볼 수 있습니다.

아래 코드는 에이전트 그래프를 import하고 구조를 시각화합니다.

In [ ]:
from agent import graph
from langchain_teddynote.graphs import visualize_graph

# 에이전트 그래프 시각화
visualize_graph(graph)

### 에이전트 실행 테스트

`stream_graph` 함수를 사용하여 에이전트에 질문을 전달하고, 도구 호출 과정과 응답을 실시간으로 확인합니다. Tavily 검색 도구가 올바르게 호출되는지, 검색 결과를 기반으로 답변이 생성되는지 검증합니다.

아래 코드는 검색이 필요한 질문을 전달하여 에이전트의 도구 호출 동작을 확인합니다.

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langchain_teddynote.messages import stream_graph, random_uuid

# config 설정
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 검색이 필요한 질문으로 에이전트 테스트
inputs = {"messages": [HumanMessage(content="2025년 AI 에이전트 트렌드에 대해 알려줘")]}

# 에이전트 스트리밍 실행
stream_graph(graph, inputs, config)

### 멀티턴 대화 테스트

`MemorySaver` 체크포인터 덕분에 동일한 `thread_id`를 사용하면 이전 대화 맥락을 유지한 채 후속 질문에 답변할 수 있습니다. 에이전트가 이전 대화 내용을 기억하고 있는지 확인합니다.

아래 코드는 동일한 thread에서 후속 질문을 전달하여 멀티턴 대화가 정상 동작하는지 테스트합니다.

In [ ]:
# 동일한 thread_id로 후속 질문 (멀티턴 대화)
inputs = {"messages": [HumanMessage(content="방금 알려준 내용을 3줄로 요약해줘")]}

# 동일한 config로 스트리밍 실행
stream_graph(graph, inputs, config)

---

## LangGraph 개발 서버 실행

에이전트가 정상적으로 동작하는 것을 확인했으므로, 이제 `langgraph dev` 명령으로 LangGraph 개발 서버를 실행합니다. 개발 서버는 에이전트를 REST API로 노출하며, Agent Chat UI와 같은 클라이언트가 이 API를 통해 에이전트와 통신합니다.

개발 서버는 기본적으로 `http://localhost:2024` 에서 실행됩니다. 코드 변경 시 자동으로 서버가 재시작되므로 개발 중에도 편리하게 사용할 수 있습니다.

아래 명령을 터미널에서 실행하세요.

In [ ]:
# 아래 명령을 터미널에서 실행하세요
print("cd 13-Deployment")
print("langgraph dev --config langgraph.json")
print()
print("서버가 시작되면 http://localhost:2024 에서 API에 접근할 수 있습니다.")
print("서버를 종료하려면 Ctrl+C를 누르세요.")

---

## Agent Chat UI 설치 및 실행

Agent Chat UI는 LangGraph 에이전트와 대화할 수 있는 웹 기반 채팅 인터페이스입니다. `npx` 명령 한 줄로 프로젝트를 생성하고 바로 실행할 수 있습니다. React 기반으로 구축되어 있으며, 도구 호출 과정을 시각적으로 표시하고 스트리밍 응답을 지원합니다.

Node.js(v18 이상)가 설치되어 있어야 합니다. 아래 명령을 **새 터미널**에서 실행하세요.

In [ ]:
# 아래 명령을 새 터미널에서 실행하세요
print("npx create-agent-chat-app")
print()
print("설치 과정에서 다음 질문에 답변하세요:")
print("  - Project name: agent-chat-ui (원하는 이름)")
print("  - Package manager: npm 또는 yarn")
print()
print("설치가 완료되면 다음 명령으로 실행합니다:")
print("  cd agent-chat-ui")
print("  npm run dev  (또는 yarn dev)")
print()
print("브라우저에서 http://localhost:3000 으로 접속합니다.")

---

## 연결 설정

Agent Chat UI를 실행하면 연결 설정 화면이 나타납니다. LangGraph 개발 서버와 연결하기 위해 다음 정보를 입력합니다. Deployment URL은 `langgraph dev` 서버 주소이고, Graph ID는 `langgraph.json`에서 정의한 그래프 이름입니다.

| 항목 | 값 |
|:---|:---|
| **Deployment URL** | `http://localhost:2024` |
| **Graph ID** | `agent` |
| **LangSmith API Key** | (선택사항) LangSmith API 키를 입력하면 추적 기능을 사용할 수 있습니다 |

연결이 성공하면 채팅 화면이 나타나고, 에이전트와 대화를 시작할 수 있습니다.

---

## 테스트 시나리오

Agent Chat UI에서 에이전트의 다양한 기능을 테스트해볼 수 있습니다. 아래 시나리오를 순서대로 실행하여 에이전트가 올바르게 동작하는지 확인하세요.

### 일반 대화

도구 호출 없이 모델이 직접 답변하는 일반 대화를 테스트합니다. 에이전트가 자연스러운 한국어로 응답하는지 확인합니다.

```
사용자: 안녕하세요! 오늘 날씨가 좋네요.
```

### 도구 호출 (웹 검색)

Tavily 검색 도구를 호출하여 최신 정보를 검색하고 답변하는 시나리오입니다. Chat UI에서 도구 호출 과정이 시각적으로 표시되는지 확인합니다.

```
사용자: 2025년 가장 인기 있는 AI 프레임워크는 무엇인가요?
```

### 멀티턴 대화

이전 대화 맥락을 유지한 채 후속 질문에 답변하는지 테스트합니다. `MemorySaver` 체크포인터를 통해 서버 측에서 대화 히스토리가 관리됩니다.

```
사용자: 그 중에서 LangGraph에 대해 더 자세히 알려줘
사용자: LangGraph의 장점을 3가지로 요약해줘
```

---

## 정리

이 튜토리얼에서는 LangGraph 에이전트를 Agent Chat UI와 연동하는 전체 과정을 다루었습니다.

1. `agent.py`에서 Tavily 검색 도구를 사용하는 ReAct 에이전트를 정의했습니다
2. `langgraph.json`으로 서버 설정을 구성했습니다
3. 노트북에서 에이전트를 로컬 테스트하여 정상 동작을 확인했습니다
4. `langgraph dev` 명령으로 개발 서버를 실행했습니다
5. Agent Chat UI를 설치하고 LangGraph 서버에 연결하여 에이전트와 대화했습니다

이 구조를 활용하면 에이전트 개발 중에 웹 UI를 통해 빠르게 테스트하고 디버깅할 수 있습니다. 에이전트의 도구를 추가하거나 프롬프트를 수정한 후 `langgraph dev` 서버가 자동으로 재시작되므로, Agent Chat UI에서 바로 변경 사항을 확인할 수 있습니다.